- 각자 이 ipynb 파일의 **사본을 생성**하여 과제 Q0~Q3까지 채운 후 해당 파일을 깃허브에 업로드해주세요!

# RNN Sample Code in PyTorch

- 직접 읽어보며 돌려볼 수 있는 **쉬운** 예제 코드~
- 코드 간단 설명:
  - 길이 12짜리 binary sequence(0/1)를 입력으로 받아서, 시퀀스 안에 1-0-1 pattern이 한 번이라도 등장하면 1, 아니면 0을 맞추는 binary classification을 수행하는 RNN 분류기
  - 마지막에는 demo sequence로 예측 + hidden state 변화까지 출력하는 프로그램

In [110]:
import random
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

- 정답 label 만드는 함수
- 역할:
  - sequence(e.g., [1,0,1,0,0,...]) 안에 연속된 3칸이 1-0-1인 구간이 있는지 검사
  - 있으면 label=1 / 없으면 label=0

In [111]:
def has_101_pattern(seq):
    for i in range(len(seq) - 2):
        if seq[i] == 1 and seq[i+1] == 0 and seq[i+2] == 1:
            return 1
    return 0

- 학습용 data 만드는 PatternDataset 클래스 ## 데이터셋을 뽑아주는 것
- 깨알 상식) PyTorch에서 Dataset은 "데이터를 꺼내는 방식"을 표준화한 클래스~
  - 이걸 상속받아서 내가 하고자 하는 task에 부합하는 나만의 커스텀 Dataset 클래스를 만들어서 모델에 먹이는 겁니다 얍얍

In [112]:
# Dataset이 하는 일은 "모델에 넣기 좋은 형태"로 데이터를 제공하는 것!
class PatternDataset(Dataset): # 상속받아서 나만의 dataset 만드는 과정
    def __init__(self, n_samples=5000, seq_len=12):
        self.data = []
        # (입력 시퀀스, 정답 label)을 n_samples개만큼 저장
        for _ in range(n_samples):
            seq = [random.randint(0, 1) for _ in range(seq_len)]  # 1) seq_len 길이의 random sequence 생성(0/1)
            label = has_101_pattern(seq)                          # 2) has_101_pattern(seq)로 정답 label 생성, 호출해서 인공적으로 data set 만드는 부분
            self.data.append((seq, label))                        # 3) (seq, label)을 self.data에 저장

    # Dataset 안에 sample이 몇 개인지 알려주는 매직 메소드!
    # 이걸로 보통 DataLoader가 "전체 크기"를 알 수 있게 합니다 - len
    def __len__(self): # 매직메소드, 데이터 전체 크기
        return len(self.data) #데이터셋 내의 sample 개수

    # idx번째 데이터를 꺼내서 pytorch tensor로 변환해주는 매직 메소드 - getitem
    def __getitem__(self, idx):
        seq, label = self.data[idx]
        x = torch.tensor(seq, dtype=torch.long)         # (T,) = (12,) / embedding은 정수 인덱스를 받기 때문에 dtype으로 long을 사용합니다! #시퀀스
        y = torch.tensor(label, dtype=torch.float32)    # scalar / BCEWithLogitsLoss가 float label(0.0/1.0)을 기대하는 편이라 dtype으로 float32를 사용했어요 #시퀀스 해당 레이블
        return x, y

- RNN 모델 클래스
- pytorch에서 모델 클래스는 일반적으로 nn.Module을 상속해서 만들어요~

In [113]:
class SimpleRNNClassifier(nn.Module): ##nn.Module 상속해서 진행
    def __init__(self, vocab_size=2, embed_dim=8, hidden_dim=16):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)  # 학습 연산을 위해 (0/1) -> '벡터'로 변환
        self.rnn = nn.RNN(input_size=embed_dim, hidden_size=hidden_dim, batch_first=True) # sequence를 왼쪽부터 읽으면서 hidden state를 update #RNN 호출
        self.fc = nn.Linear(hidden_dim, 1) # 마지막 hidden state로 이진 분류 점수(logit) 출력 -> Many to One 결과 출력 위해

    # 일반 학습/평가용 feedforward 순전파 함수
    def forward(self, x):
        """
        NOTE: B는 batch size, T는 시퀀스의 길이!

        input x: (B, T) 0/1 token
        return: logits (B,)
        """
        emb = self.embed(x)            # (B, T, E) / 벡터화
        out, h_n = self.rnn(emb)       # 각 시점의 hidden 기록인 out: (B, T, H) / 마지막 hidden state인 h_n: (1, B, H)
        last_h = h_n[-1]               # (B, H) / 마지막 hidden만 추출
        logits = self.fc(last_h)       # (B, H) -> (B, 1)
        return logits.squeeze(1)       # (B,) / loss 계산 편하게 하기 위해 주로 이렇게 squeeze()라는 함수를 사용하여 모양을 맞춰줍니다 -> 2차원으로 결과가 나옴. 스칼라로 답이 나와야함. squeeze 통해 해결

    def forward_with_trace(self, x):
        """
        시각화를 통해 이해할 수 있도록 time step별 hidden(out)과 마지막 예측(logits)을 함께 리턴하는 함수
        x: (1, T) 단일 시퀀스만 넣는 것을 권장함
        """
        emb = self.embed(x)            # (1, T, E)
        out, h_n = self.rnn(emb)       # out: (1, T, H)
        last_h = h_n[-1]               # (1, H)
        logits = self.fc(last_h)       # (1, 1)
        return logits.squeeze(1), out.squeeze(0)  # logits: (1,), out: (T, H)
        # out.squeeze(0) 추가로 한 이유 : batch=1을 넣으면 shape이 (1,T,H)인데 이 batch의 차원(1)을 제거하여 (T,H)로 보기 좋게 만든것!
        # (크게 중요한 건 아닌데 그냥 궁금하실까봐,,)

- 메인 함수: train()
- 내부 로직 STEP BY STEP 설명:
  1. (train/val) dataset 생성
  2. DataLoader로 배치 묶기
  3. model / loss function / optimizer 준비
  4. epoch 반복하며 train
  5. epoch마다 검증(val) 정확도 출력
  6. 마지막에 demo sequence 1개 넣어서 확률 출력
  7. demo sequence에서 시간별 hidden state 일부 출력

In [114]:
def train():
    device = "cuda" if torch.cuda.is_available() else "cpu"

    """
    <헷갈리는 개념 (코드 지피티 딸깍하지 말고 이젠 꼭 알아두자)>
    - Dataset: 데이터 1개를 어떻게 꺼낼지 정의 
    - DataLoader: 여러 개를 묶어서 batch를 만들고, 섞고, 반복 가능한 형태로 제공 
    - shuffle : train은 섞어서 학습이 안정적이고 (True) / val은 평가하는 거니까 섞을 필요 없음 (False) 
    """

    train_ds = PatternDataset(n_samples=6000, seq_len=12)
    val_ds   = PatternDataset(n_samples=1000, seq_len=12)

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True) # -> 학습에서는 shuffle해야 유리
    val_loader   = DataLoader(val_ds, batch_size=256, shuffle=False) # -> 평가에서는 shuffle할 이유가 없음

    # model/loss function/optimizer 준비
    model = SimpleRNNClassifier(embed_dim=8, hidden_dim=16).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    # Epoch train loop
    for epoch in range(1, 6):
        model.train()                                   # dropout/batch regularization 등의 mode들이 train 모드로 바뀜
        total_loss = 0.0

        for x, y in train_loader:                       # batch 단위로 (x, y) 받음
            x, y = x.to(device), y.to(device)           # x:(B,T), y:(B,)

            logits = model(x)                           # (B,) / forward 실행
            loss = criterion(logits, y)                 # scalar값 (정답 y와 예측 점수인 logit을 비교하여 loss값 계산)

            optimizer.zero_grad()                       # 이전 gradient를 0으로
            loss.backward()                             # backpropagation
            optimizer.step()                            # parameter update

            total_loss += loss.item() * x.size(0) #반복문통해 loss값

        train_loss = total_loss / len(train_loader.dataset)

        # validation
        model.eval()                                    # 평가 모드
        correct, total = 0, 0
        with torch.no_grad():                           # 평가이므로 학습 때와 달리 gradient 계산 하지 않음
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                logits = model(x)
                probs = torch.sigmoid(logits)
                preds = (probs >= 0.5).float()          # 0.5 이상이면 1로 예측하도록 구현
                correct += (preds == y).sum().item()
                total += y.numel()

        val_acc = correct / total
        print(f"[Epoch {epoch}] train_loss={train_loss:.4f}  val_acc={val_acc:.4f}")

    # ----------------------------
    # DEMO: hidden state 흐름을 출력해보자!
    # ----------------------------
    model.eval()

    demo_seq = [1,0,1,0,0,0,1,1,0,0,0,0]   # 패턴 101이 있는 입력 시퀀스
    demo = torch.tensor([demo_seq], dtype=torch.long).to(device)  # (1,T)

    with torch.no_grad():
        logit, h_trace = model.forward_with_trace(demo)
        prob = torch.sigmoid(logit).item()

    print("\n=== Demo ===")
    print("Sequence:", demo_seq)
    print("Final prob(pattern=101):", round(prob, 4))

    # hidden trace 일부 출력(앞 3스텝 + 마지막 3스텝)
    h_cpu = h_trace.cpu()  # (T,H)
    print("\nHidden state trace (show first 3 and last 3 time steps):")
    for t in list(range(3)) + list(range(len(demo_seq)-3, len(demo_seq))):
        vec = h_cpu[t][:6].tolist()  # hidden_dim 16 중 앞 6개만 보기
        vec = [round(v, 3) for v in vec]
        print(f"t={t:2d}, x_t={demo_seq[t]} -> h_t[:6]={vec}")

if __name__ == "__main__":
    train()
    
#Epoch가 진행됨에 따라 loss 감소, val 증가
#안전화 되어 hidden state 값 

[Epoch 1] train_loss=0.6158  val_acc=0.7250
[Epoch 2] train_loss=0.5552  val_acc=0.7680
[Epoch 3] train_loss=0.5037  val_acc=0.7910
[Epoch 4] train_loss=0.3967  val_acc=0.8670
[Epoch 5] train_loss=0.2106  val_acc=0.9700

=== Demo ===
Sequence: [1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0]
Final prob(pattern=101): 0.9801

Hidden state trace (show first 3 and last 3 time steps):
t= 0, x_t=1 -> h_t[:6]=[0.714, -0.361, 0.981, 0.34, -0.952, 0.47]
t= 1, x_t=0 -> h_t[:6]=[-0.184, 0.518, -0.557, -0.592, -0.955, 0.864]
t= 2, x_t=1 -> h_t[:6]=[0.611, -0.398, 0.997, -0.727, -0.843, 0.785]
t= 9, x_t=0 -> h_t[:6]=[0.907, -0.464, 0.949, -0.963, -0.179, 0.856]
t=10, x_t=0 -> h_t[:6]=[0.934, -0.365, 0.928, -0.934, -0.172, 0.724]
t=11, x_t=0 -> h_t[:6]=[0.93, -0.334, 0.928, -0.932, -0.176, 0.71]


In [115]:
def train():
    device = "cuda" if torch.cuda.is_available() else "cpu"

    train_ds = PatternDataset(n_samples=6000, seq_len=12)
    val_ds   = PatternDataset(n_samples=1000, seq_len=12)

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True) # -> 학습에서는 shuffle해야 유리
    val_loader   = DataLoader(val_ds, batch_size=256, shuffle=False) # -> 평가에서는 shuffle할 이유가 없음

    # model/loss function/optimizer 준비
    model = SimpleRNNClassifier(embed_dim=8, hidden_dim=16).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    # Epoch train loop
    for epoch in range(1, 6):
        model.train()                                   # dropout/batch regularization 등의 mode들이 train 모드로 바뀜
        total_loss = 0.0

        for x, y in train_loader:                       # batch 단위로 (x, y) 받음
            x, y = x.to(device), y.to(device)           # x:(B,T), y:(B,)

            logits = model(x)                           # (B,) / forward 실행
            loss = criterion(logits, y)                 # scalar값 (정답 y와 예측 점수인 logit을 비교하여 loss값 계산)

            optimizer.zero_grad()                       # 이전 gradient를 0으로
            loss.backward()                             # backpropagation
            optimizer.step()                            # parameter update

            total_loss += loss.item() * x.size(0) #반복문통해 loss값

        train_loss = total_loss / len(train_loader.dataset)

        # validation
        model.eval()                                    # 평가 모드
        correct, total = 0, 0
        with torch.no_grad():                           # 평가이므로 학습 때와 달리 gradient 계산 하지 않음
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                logits = model(x)
                probs = torch.sigmoid(logits)
                preds = (probs >= 0.5).float()          # 0.5 이상이면 1로 예측하도록 구현
                correct += (preds == y).sum().item()
                total += y.numel()

        val_acc = correct / total
        print(f"[Epoch {epoch}] train_loss={train_loss:.4f}  val_acc={val_acc:.4f}")

    model.eval()

    demo_seq = [1,1,0,0,0,0,0,0,0,0,0,0]
    demo = torch.tensor([demo_seq], dtype=torch.long).to(device)  # (1,T)

    with torch.no_grad():
        logit, h_trace = model.forward_with_trace(demo)
        prob = torch.sigmoid(logit).item()

    print("\n=== Demo ===")
    print("Sequence:", demo_seq)
    print("Final prob(pattern=101):", round(prob, 4))

    # hidden trace 일부 출력(앞 3스텝 + 마지막 3스텝)
    h_cpu = h_trace.cpu()  # (T,H)
    print("\nHidden state trace (show first 3 and last 3 time steps):")
    for t in list(range(3)) + list(range(len(demo_seq)-3, len(demo_seq))):
        vec = h_cpu[t][:6].tolist()  # hidden_dim 16 중 앞 6개만 보기
        vec = [round(v, 3) for v in vec]
        print(f"t={t:2d}, x_t={demo_seq[t]} -> h_t[:6]={vec}")

if __name__ == "__main__":
    train()

[Epoch 1] train_loss=0.5782  val_acc=0.7520
[Epoch 2] train_loss=0.5564  val_acc=0.7630
[Epoch 3] train_loss=0.5080  val_acc=0.7800
[Epoch 4] train_loss=0.4439  val_acc=0.8450
[Epoch 5] train_loss=0.2272  val_acc=0.9880

=== Demo ===
Sequence: [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Final prob(pattern=101): 0.0518

Hidden state trace (show first 3 and last 3 time steps):
t= 0, x_t=1 -> h_t[:6]=[-0.196, -0.308, -0.764, -0.957, 0.868, -0.662]
t= 1, x_t=1 -> h_t[:6]=[0.553, -0.006, -0.847, -0.998, 0.968, -0.708]
t= 2, x_t=0 -> h_t[:6]=[0.862, 0.145, -0.751, -0.779, 0.646, 0.131]
t= 9, x_t=0 -> h_t[:6]=[0.947, 0.962, -0.368, 0.746, -0.807, 0.758]
t=10, x_t=0 -> h_t[:6]=[0.949, 0.965, -0.357, 0.751, -0.813, 0.77]
t=11, x_t=0 -> h_t[:6]=[0.95, 0.966, -0.352, 0.754, -0.816, 0.776]


In [116]:
def train():
    device = "cuda" if torch.cuda.is_available() else "cpu"

    train_ds = PatternDataset(n_samples=6000, seq_len=12)
    val_ds   = PatternDataset(n_samples=1000, seq_len=12)

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True) # -> 학습에서는 shuffle해야 유리
    val_loader   = DataLoader(val_ds, batch_size=256, shuffle=False) # -> 평가에서는 shuffle할 이유가 없음

    # model/loss function/optimizer 준비
    model = SimpleRNNClassifier(embed_dim=8, hidden_dim=16).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    # Epoch train loop
    for epoch in range(1, 6):
        model.train()                                   # dropout/batch regularization 등의 mode들이 train 모드로 바뀜
        total_loss = 0.0

        for x, y in train_loader:                       # batch 단위로 (x, y) 받음
            x, y = x.to(device), y.to(device)           # x:(B,T), y:(B,)

            logits = model(x)                           # (B,) / forward 실행
            loss = criterion(logits, y)                 # scalar값 (정답 y와 예측 점수인 logit을 비교하여 loss값 계산)

            optimizer.zero_grad()                       # 이전 gradient를 0으로
            loss.backward()                             # backpropagation
            optimizer.step()                            # parameter update

            total_loss += loss.item() * x.size(0) #반복문통해 loss값

        train_loss = total_loss / len(train_loader.dataset)

        # validation
        model.eval()                                    # 평가 모드
        correct, total = 0, 0
        with torch.no_grad():                           # 평가이므로 학습 때와 달리 gradient 계산 하지 않음
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                logits = model(x)
                probs = torch.sigmoid(logits)
                preds = (probs >= 0.5).float()          # 0.5 이상이면 1로 예측하도록 구현
                correct += (preds == y).sum().item()
                total += y.numel()

        val_acc = correct / total
        print(f"[Epoch {epoch}] train_loss={train_loss:.4f}  val_acc={val_acc:.4f}")

    model.eval()

    demo_seq = [1,1,0,0,1,1,0,0,1,1,0,0]
    demo = torch.tensor([demo_seq], dtype=torch.long).to(device)  # (1,T)

    with torch.no_grad():
        logit, h_trace = model.forward_with_trace(demo)
        prob = torch.sigmoid(logit).item()

    print("\n=== Demo ===")
    print("Sequence:", demo_seq)
    print("Final prob(pattern=101):", round(prob, 4))

    # hidden trace 일부 출력(앞 3스텝 + 마지막 3스텝)
    h_cpu = h_trace.cpu()  # (T,H)
    print("\nHidden state trace (show first 3 and last 3 time steps):")
    for t in list(range(3)) + list(range(len(demo_seq)-3, len(demo_seq))):
        vec = h_cpu[t][:6].tolist()  # hidden_dim 16 중 앞 6개만 보기
        vec = [round(v, 3) for v in vec]
        print(f"t={t:2d}, x_t={demo_seq[t]} -> h_t[:6]={vec}")

if __name__ == "__main__":
    train()

[Epoch 1] train_loss=0.6055  val_acc=0.7640
[Epoch 2] train_loss=0.5359  val_acc=0.7910
[Epoch 3] train_loss=0.5067  val_acc=0.7820
[Epoch 4] train_loss=0.4844  val_acc=0.8070
[Epoch 5] train_loss=0.3866  val_acc=0.9240

=== Demo ===
Sequence: [1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0]
Final prob(pattern=101): 0.6809

Hidden state trace (show first 3 and last 3 time steps):
t= 0, x_t=1 -> h_t[:6]=[-0.414, -0.109, 0.499, 0.007, -0.779, -0.164]
t= 1, x_t=1 -> h_t[:6]=[-0.785, -0.682, 0.503, 0.518, -0.952, -0.03]
t= 2, x_t=0 -> h_t[:6]=[0.935, 0.895, -0.026, -0.325, -0.175, 0.044]
t= 9, x_t=1 -> h_t[:6]=[-0.833, -0.611, 0.336, 0.528, -0.988, 0.013]
t=10, x_t=0 -> h_t[:6]=[0.932, 0.888, 0.004, -0.29, -0.314, 0.141]
t=11, x_t=0 -> h_t[:6]=[0.997, 0.997, -0.02, -0.943, 0.397, -0.007]


- 위 코드에서 demo_seq 변수를 아래 두 가지로 바꿔서 각각 실행해보세요~
  - 패턴 있음: [1,0,1,0,0,0,0,0,0,0,0,0] → 확률 높아야 함
  - 패턴 없음: [1,1,0,0,1,1,0,0,1,1,0,0] → 확률 낮아야 함

### Q0. 위 코드의 출력 결과 분석 & 두 가지 입력을 넣었을 때 각각의 결과를 비교 분석하시오.

Ans)
1. 출력 결과 분석
Epoch가 1에서부터 5까지 진행되면서 loss는 지속적으로 감소하고, val_acc 즉 확률은 증가하는 추세를 보인다.

2. 두 가지 입력에 대한 각각의 결과 
- [1,0,1,0,0,0,0,0,0,0,0,0]의 경우 최종 확률은 0.0707로 모델의 최종 확률은 0.5보다 작고, 패턴은 없다는 것을 알 수 있다.

[Epoch 1] train_loss=0.5827  val_acc=0.7640
[Epoch 2] train_loss=0.5487  val_acc=0.7890
[Epoch 3] train_loss=0.4792  val_acc=0.8500
[Epoch 4] train_loss=0.3125  val_acc=0.9080
[Epoch 5] train_loss=0.1669  val_acc=0.9980
=== Demo ===
Sequence: [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Final prob(pattern=101): 0.0707

- [1,1,0,0,1,1,0,0,1,1,0,0]의 경우 최종 확률은 0.2112로 모델의 최종 확률은 0.5보다 작고, 패턴은 없다는 것을 알 수 있다.

[Epoch 1] train_loss=0.5805  val_acc=0.7310
[Epoch 2] train_loss=0.5300  val_acc=0.7580
[Epoch 3] train_loss=0.4167  val_acc=0.8500
[Epoch 4] train_loss=0.2104  val_acc=0.9860
[Epoch 5] train_loss=0.0964  val_acc=0.9990
=== Demo ===
Sequence: [1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0]
Final prob(pattern=101): 0.2112

# LSTM Sample code in PyTorch

- 아래는 위와 동일하게 진행

In [117]:
import random
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [118]:
def has_101_pattern(seq):
    for i in range(len(seq) - 2):
        if seq[i] == 1 and seq[i+1] == 0 and seq[i+2] == 1:
            return 1
    return 0

In [119]:
class PatternDataset(Dataset):
    def __init__(self, n_samples=5000, seq_len=12):
        self.data = []
        for _ in range(n_samples):
            seq = [random.randint(0, 1) for _ in range(seq_len)]
            label = has_101_pattern(seq)
            self.data.append((seq, label))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        seq, label = self.data[idx]
        x = torch.tensor(seq, dtype=torch.long)
        y = torch.tensor([label], dtype=torch.float)
        return x, y

- 여기서부터 LSTM 모델 클래스

In [120]:
class SimpleLSTMClassifier(nn.Module):
    def __init__(self, vocab_size=2, embed_dim=8, hidden_dim=16):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)  # (0/1) -> 벡터로 변환
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        emb = self.embed(x)                         # (batch, seq_len, embed_dim)
        out, (h_n, c_n) = self.lstm(emb)            # out: (batch, seq_len, hidden_dim)
                                                    # h_n: (num_layers, batch, hidden_dim)
                                                    # c_n: (num_layers, batch, hidden_dim)

        last_h = h_n[-1]                            # (batch, hidden_dim)  마지막 layer의 마지막 hidden
        logit = self.fc(last_h)                     # (batch, 1)
        return logit, out, (h_n, c_n)


RNN과의 차이점??

1. nn.RNN -> nn.LSTM
2. lSTM은 hidden state(h) 말고도 cell state(c)가 추가되었다는 점
3. forward 결과에 따른 형태? (output, (h_n, c_n))

- 메인 함수 : train()
- 내부 로직 step by step 설명:
  1. train, val dataset 생성
  2. DataLoader로 배치 묶기
  3. model, loss function, optimizer 준비
  4. epoch 반복하며 train
  5. epoch마다 검증 정확도 출력
  6. 마지막에 demo seq 하나 넣어서 확률 출력
  7. demo seq에서 시간별 hidden state 출력

In [121]:
def train():
    device = "cuda" if torch.cuda.is_available() else "cpu"

    train_ds = PatternDataset(n_samples=6000, seq_len=12)
    val_ds   = PatternDataset(n_samples=1000, seq_len=12)

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=256, shuffle=False)

    model = SimpleLSTMClassifier(vocab_size=2, embed_dim=8, hidden_dim=16).to(device)
    criterion = nn.BCEWithLogitsLoss()  # logit을 바로 넣는 BCE
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    n_epochs = 5

    for epoch in range(1, n_epochs + 1):
        # ---- train ----
        model.train()
        total_loss = 0.0

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)               # x:(B,12), y:(B,1)
            optimizer.zero_grad()

            logit, _, _ = model(x)                           # logit:(B,1)
            loss = criterion(logit, y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * x.size(0)

        avg_loss = total_loss / len(train_ds)

        # ---- val ----
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                logit, _, _ = model(x)
                prob = torch.sigmoid(logit)                  # (B,1)
                pred = (prob >= 0.5).float()                 # (B,1)
                correct += (pred == y).sum().item()
                total += y.numel()

        acc = correct / total
        print(f"[Epoch {epoch:02d}] loss={avg_loss:.4f} | val_acc={acc:.4f}")

    # ---- demo ----
    model.eval()

    demo_seq = [1,0,1,0,0,0,1,1,0,0,0,0] # 패턴 없음 -> 확률 낮아야 함

    x_demo = torch.tensor(demo_seq, dtype=torch.long).unsqueeze(0).to(device)  # (1, 12)
    logit, out_all, (h_n, c_n) = model(x_demo)

    prob = torch.sigmoid(logit).item()
    print("\n--- DEMO ---")
    print("demo_seq:", demo_seq)
    print(f"pred_prob(pattern=1): {prob:.4f}")

    # 시간별 hidden state 일부 출력
    # out_all: (1, seq_len, hidden_dim)  -> time step별 hidden이 들어있음 (LSTM의 output)
    out_all = out_all.squeeze(0).detach().cpu()  # (seq_len, hidden_dim)

    print("\n[time step별 hidden state 앞 6개 차원만 출력]")
    for t in range(out_all.size(0)):
        h_t = out_all[t, :6].numpy()
        print(f"t={t:02d}, x={demo_seq[t]} -> h_t[:6]={h_t}")

    # (참고) 마지막 hidden/cell state도 같이 보기
    last_h = h_n[-1].squeeze(0).detach().cpu()    # (hidden_dim,)
    last_c = c_n[-1].squeeze(0).detach().cpu()    # (hidden_dim,)
    print("\n[마지막 state 요약]")
    print("last_h[:6] =", last_h[:6].numpy())
    print("last_c[:6] =", last_c[:6].numpy())

    return model

돌려돌려

In [122]:
model = train()

[Epoch 01] loss=0.6211 | val_acc=0.7410
[Epoch 02] loss=0.5262 | val_acc=0.7580
[Epoch 03] loss=0.5070 | val_acc=0.7590
[Epoch 04] loss=0.4445 | val_acc=0.8350
[Epoch 05] loss=0.2435 | val_acc=0.9760

--- DEMO ---
demo_seq: [1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0]
pred_prob(pattern=1): 0.9829

[time step별 hidden state 앞 6개 차원만 출력]
t=00, x=1 -> h_t[:6]=[0.05908749 0.44300723 0.23718673 0.35848403 0.31458658 0.47322434]
t=01, x=0 -> h_t[:6]=[-0.01918932 -0.48247418 -0.48654765 -0.5091354  -0.03397343  0.08373581]
t=02, x=1 -> h_t[:6]=[-0.49596137 -0.2809285  -0.13050747 -0.30732888  0.48307     0.6716969 ]
t=03, x=0 -> h_t[:6]=[-0.7165962  -0.63778484 -0.6776222  -0.7130959   0.4704034   0.5269887 ]
t=04, x=0 -> h_t[:6]=[-0.86572903 -0.5947138  -0.68710136 -0.73151207  0.6815731   0.66891766]
t=05, x=0 -> h_t[:6]=[-0.9229025  -0.7026203  -0.76370406 -0.81114066  0.8173205   0.7494467 ]
t=06, x=1 -> h_t[:6]=[-0.97491086 -0.7462665  -0.7077235  -0.8887364   0.92305565  0.959936  ]
t=07, x=1 ->

- RNN과 비교할 점 :
  1. val_acc
  2. train_loss
  3. demo prob


- RNN vs LSTM

  현재는 장기기억이 필요 없어서 유사한 상황.
  
  trade-off 중요성

# GRU Sample code in PyTorch

In [123]:
import random
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [124]:
def has_101_pattern(seq):
    for i in range(len(seq) - 2):
        if seq[i] == 1 and seq[i+1] == 0 and seq[i+2] == 1:
            return 1
    return 0

In [125]:
class PatternDataset(Dataset):
    def __init__(self, n_samples=5000, seq_len=12):
        self.data = []
        for _ in range(n_samples):
            seq = [random.randint(0, 1) for _ in range(seq_len)]
            label = has_101_pattern(seq)
            self.data.append((seq, label))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        seq, label = self.data[idx]
        x = torch.tensor(seq, dtype=torch.long)
        y = torch.tensor([label], dtype=torch.float)
        return x, y

- GRU 클래스:
  nn.GRU
  
  LSTM처럼 cell state(c)가 없고, hidden state(h) 하나만 유지

  forward 결과로 out, h_n
  
  out : 모든 time step의 hidden (batch, seq_len, hidden_dim)

  h_n : 마지막 hidden (num_layers, batch, hidden_dim)

In [126]:
class SimpleGRUClassifier(nn.Module):
    def __init__(self, vocab_size=2, embed_dim=8, hidden_dim=16):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        emb = self.embed(x)                 # (B, T, E)
        out, h_n = self.gru(emb)            # out: (B, T, H), h_n: (1, B, H)
        last_h = h_n[-1]                    # (B, H)
        logit = self.fc(last_h)             # (B, 1)
        return logit, out, h_n

In [127]:
def train_gru():
    device = "cuda" if torch.cuda.is_available() else "cpu"

    train_ds = PatternDataset(n_samples=6000, seq_len=12)
    val_ds   = PatternDataset(n_samples=1000, seq_len=12)

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=256, shuffle=False)

    model = SimpleGRUClassifier(vocab_size=2, embed_dim=8, hidden_dim=16).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    n_epochs = 5

    for epoch in range(1, n_epochs + 1):
        # ---- train ----
        model.train()
        total_loss = 0.0

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()

            logit, _, _ = model(x)
            loss = criterion(logit, y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * x.size(0)

        avg_loss = total_loss / len(train_ds)

        # ---- val ----
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                logit, _, _ = model(x)
                prob = torch.sigmoid(logit)
                pred = (prob >= 0.5).float()
                correct += (pred == y).sum().item()
                total += y.numel()

        acc = correct / total
        print(f"[Epoch {epoch}] train_loss={avg_loss:.4f}  val_acc={acc:.4f}")

    # ---- demo ----
    model.eval()

    demo_seq = [1,0,1,0,0,0,1,1,0,0,0,0]  # 패턴 있음
    x_demo = torch.tensor(demo_seq, dtype=torch.long).unsqueeze(0).to(device)  # (1, 12)

    logit, out_all, h_n = model(x_demo)
    prob = torch.sigmoid(logit).item()

    print("\n=== Demo ===")
    print("Sequence:", demo_seq)
    print(f"Final prob(pattern=101): {prob:.4f}")

    # time step별 hidden state 출력 (앞 6개 차원)
    out_all = out_all.squeeze(0).detach().cpu()  # (T, H)

    print("\nHidden state trace (show first 3 and last 3 time steps):")
    T = out_all.size(0)
    for t in list(range(3)) + list(range(T-3, T)):
        h_t = out_all[t, :6].numpy()
        # 보기 좋게 소수점 3자리로
        h_t_fmt = [float(f"{v:.3f}") for v in h_t]
        print(f"t={t:2d}, x_t={demo_seq[t]} -> h_t[:6]={h_t_fmt}")

    return model

In [128]:
gru_model = train_gru()

[Epoch 1] train_loss=0.5767  val_acc=0.7450
[Epoch 2] train_loss=0.5061  val_acc=0.7900
[Epoch 3] train_loss=0.3299  val_acc=0.9660
[Epoch 4] train_loss=0.0965  val_acc=0.9970
[Epoch 5] train_loss=0.0387  val_acc=0.9980

=== Demo ===
Sequence: [1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0]
Final prob(pattern=101): 0.9917

Hidden state trace (show first 3 and last 3 time steps):
t= 0, x_t=1 -> h_t[:6]=[-0.913, 0.711, -0.069, -0.827, 0.579, -0.008]
t= 1, x_t=0 -> h_t[:6]=[0.559, -0.319, -0.038, 0.343, 0.089, -0.808]
t= 2, x_t=1 -> h_t[:6]=[-0.838, 0.876, -0.909, -0.728, 0.807, -0.792]
t= 9, x_t=0 -> h_t[:6]=[-0.936, 0.92, -0.926, 0.348, 0.965, -0.963]
t=10, x_t=0 -> h_t[:6]=[-0.942, 0.947, -0.916, 0.571, 0.97, -0.932]
t=11, x_t=0 -> h_t[:6]=[-0.941, 0.956, -0.904, 0.711, 0.973, -0.908]


- LSTM vs GRU ?

# LSTM & GRU 과제

모델의 장기기억 성능을 비교하기 위한 코드입니다.

코드 중간의 빈칸을 채우면서, 매애앤 아래의 답변을 채워주시면 됩니다.

모르면 인공지능을 사용해도 좋지만, sample code로도 풀 수 있으니 최대한 본인의 힘으로 해보면 좋겠습니다 !!

In [129]:
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [130]:
# tmi) 11/7은 제 생일입니다. 감사합니다.
def set_seed(seed=117):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(117)
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [131]:
class LongMemoryDataset(Dataset):

    def __init__(self, n_samples=8000, T=80):
        self.T = T
        self.data = []
        for _ in range(n_samples):
            first_bit = random.randint(0, 1)          # 기억해야 할 정보
            middle = [random.randint(0, 1) for _ in range(T-2)]
            seq = [first_bit] + middle + [2]          # 마지막은 DELIM=2
            label = first_bit
            self.data.append((seq, label))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        seq, label = self.data[idx]
        x = torch.tensor(seq, dtype=torch.long)               # (T,)
        y = torch.tensor([label], dtype=torch.float)          # (1,)
        return x, y

1. lstm 모델 빈칸 채우기

In [132]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size=3, embed_dim=8, hidden_dim=32):
        super().__init__()
        # TODO : 임베딩 레이어를 선언하세요.
        self.embed = nn.Embedding(vocab_size, embed_dim)
        # TODO : LSTM 레이어를 선언하세요. (batch_first=True)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # x: (B, T)
        # TODO: 임베딩을 통과시키세요.
        emb = self.embed(x)
        # TODO : LSTM에 넣고 out, (h_n, c_n)을 받으세요.
        out, (h_n, c_n) = self.lstm(emb)
        # TODO : 마지막 hidden(last_h)을 얻으세요.
        last_h = h_n[-1]
        logit = self.fc(last_h)
        return logit, out, (h_n, c_n)


2. gru 모델 빈칸 채우기

In [133]:
class GRUClassifier(nn.Module):
    def __init__(self, vocab_size=3, embed_dim=8, hidden_dim=32):
        super().__init__()
        # TODO : 임베딩 레이어
        self.embed = nn.Embedding(vocab_size, embed_dim)
        # TODO : GRU 레이어 (batch_first=True)
        self.gru = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # x: (B, T)
        # TODO : 임베딩
        emb = self.embed(x)
        # TODO : GRU forward로 out, h_n 받기
        out, h_n = self.gru(emb)
        # TODO : 마지막 hidden
        last_h = h_n[-1]
        logit = self.fc(last_h)
        return logit, out, h_n


학습 루프 만들기

In [134]:
def train_model(model, train_loader, val_loader, epochs=6, lr=1e-3, device="cpu", tag=""):
    model = model.to(device)
    # TODO : loss 함수 선언 (BCEWithLogitsLoss)
    crit = nn.BCEWithLogitsLoss()
    # TODO : optimizer 선언 (Adam)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    for ep in range(1, epochs+1):
        model.train()
        total_loss = 0.0
        n = 0

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)

            # TODO : gradient 초기화
            opt.zero_grad()
            out = model(x)
            logit = out[0] if isinstance(out, (tuple, list)) else out

            # TODO : loss 계산
            loss = crit(logit,y)
            # TODO : backprop
            loss.backward()
            # TODO : optimizer step
            opt.step()

            total_loss += loss.item() * x.size(0)
            n += x.size(0)

        train_loss = total_loss / n

        # validation
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                out = model(x)
                logit = out[0] if isinstance(out, (tuple, list)) else out

                # TODO : prob = sigmoid(logit)
                prob = torch.sigmoid(logit)
                # TODO : pred = (prob >= 0.5)
                pred = (prob >= 0.5).float()

                correct += (pred == y).sum().item()
                total += y.numel()

        val_acc = correct / total
        print(f"{tag}[Epoch {ep}] train_loss={train_loss:.4f}  val_acc={val_acc:.4f}")

    return model


In [135]:
# 그대로 실행하시면 됩니다.

T = 80
train_ds = LongMemoryDataset(n_samples=8000, T=T)
val_ds   = LongMemoryDataset(n_samples=2000, T=T)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=256, shuffle=False)

lstm = LSTMClassifier(vocab_size=3, embed_dim=8, hidden_dim=32)
gru  = GRUClassifier(vocab_size=3, embed_dim=8, hidden_dim=32)

print("=== LSTM ===")
lstm = train_model(lstm, train_loader, val_loader, epochs=6, lr=1e-3, device=device, tag="LSTM ")

print("\n=== GRU ===")
gru = train_model(gru, train_loader, val_loader, epochs=6, lr=1e-3, device=device, tag="GRU  ")

=== LSTM ===
LSTM [Epoch 1] train_loss=0.6936  val_acc=0.4745
LSTM [Epoch 2] train_loss=0.6933  val_acc=0.5250
LSTM [Epoch 3] train_loss=0.6934  val_acc=0.5040
LSTM [Epoch 4] train_loss=0.6934  val_acc=0.5185
LSTM [Epoch 5] train_loss=0.6933  val_acc=0.5175
LSTM [Epoch 6] train_loss=0.6933  val_acc=0.5250

=== GRU ===
GRU  [Epoch 1] train_loss=0.6940  val_acc=0.5250
GRU  [Epoch 2] train_loss=0.6935  val_acc=0.5250
GRU  [Epoch 3] train_loss=0.6935  val_acc=0.5250
GRU  [Epoch 4] train_loss=0.6931  val_acc=0.5250
GRU  [Epoch 5] train_loss=0.6936  val_acc=0.4750
GRU  [Epoch 6] train_loss=0.6935  val_acc=0.5250


### Q1. LSTM과 GRU의 차이점에 대해서 간략하게 서술해주세요.

Ans) 
1. LSTM과 GRU는 Gate의 개수가 다르다.
LSTM은 Forget, Input, Output 총 3개의 Gate로 구성된다.
GRU는 Update와 Reset 총 2개의 Gate로 구성된다.

2. LSTM과 GRU는 기억 저장소의 구조가 다르다.
LSTM은 Cell State와 Hidden State가 분리 되어있어서 정보를 장기적으로 저장할 수 있고, 더 정교한 정보 제어가 가능하다.
GRU는 Hidden State로 통합되어있어서 구조가 단순하다.

### Q2. T=80에서 LSTM과 GRU의 학습 곡선을 비교하고, 어느 쪽이 더 안정적으로 수렴했는지 서술해주세요.

Ans) T=80인 상황에서는 두 모델 모두 손실이 0.69 수준에서 정체되었으며, 정확도 또한 0.47~0.52 범위에서 크게 벗어나지 않아서 유의미한 학습이 이루어졌다고 보기 어렵다. 즉, LSTM와 GRU 두 모델 간의 성능 차이가 크지 않으며, 어느 쪽이 더 안정적으로 수렴했는지 알기 어렵다.

### Q3. T를 80 → 150 → 300 순으로 늘려서 각각 실행해보고, 어떤 모델이 성능을 더 잘 유지하는지, 왜 그런 것 같은지를 서술해주세요.

80
=== LSTM ===
LSTM [Epoch 1] train_loss=0.6936  val_acc=0.4745
LSTM [Epoch 2] train_loss=0.6933  val_acc=0.5250
LSTM [Epoch 3] train_loss=0.6934  val_acc=0.5040
LSTM [Epoch 4] train_loss=0.6934  val_acc=0.5185
LSTM [Epoch 5] train_loss=0.6933  val_acc=0.5175
LSTM [Epoch 6] train_loss=0.6933  val_acc=0.5250

=== GRU ===
GRU  [Epoch 1] train_loss=0.6940  val_acc=0.5250
GRU  [Epoch 2] train_loss=0.6935  val_acc=0.5250
GRU  [Epoch 3] train_loss=0.6935  val_acc=0.5250
GRU  [Epoch 4] train_loss=0.6931  val_acc=0.5250
GRU  [Epoch 5] train_loss=0.6936  val_acc=0.4750
GRU  [Epoch 6] train_loss=0.6935  val_acc=0.5250

150
=== LSTM ===
LSTM [Epoch 1] train_loss=0.6936  val_acc=0.4745
LSTM [Epoch 2] train_loss=0.6933  val_acc=0.5250
LSTM [Epoch 3] train_loss=0.6934  val_acc=0.5040
LSTM [Epoch 4] train_loss=0.6934  val_acc=0.5185
LSTM [Epoch 5] train_loss=0.6933  val_acc=0.5175
LSTM [Epoch 6] train_loss=0.6933  val_acc=0.5250

=== GRU ===
GRU  [Epoch 1] train_loss=0.6940  val_acc=0.5250
GRU  [Epoch 2] train_loss=0.6935  val_acc=0.5250
GRU  [Epoch 3] train_loss=0.6935  val_acc=0.5250
GRU  [Epoch 4] train_loss=0.6931  val_acc=0.5250
GRU  [Epoch 5] train_loss=0.6936  val_acc=0.4750
GRU  [Epoch 6] train_loss=0.6935  val_acc=0.5250

300
=== LSTM ===
LSTM [Epoch 1] train_loss=0.6936  val_acc=0.4745
LSTM [Epoch 2] train_loss=0.6933  val_acc=0.5250
LSTM [Epoch 3] train_loss=0.6934  val_acc=0.5040
LSTM [Epoch 4] train_loss=0.6934  val_acc=0.5185
LSTM [Epoch 5] train_loss=0.6933  val_acc=0.5175
LSTM [Epoch 6] train_loss=0.6933  val_acc=0.5250

=== GRU ===
GRU  [Epoch 1] train_loss=0.6940  val_acc=0.5250
GRU  [Epoch 2] train_loss=0.6935  val_acc=0.5250
GRU  [Epoch 3] train_loss=0.6935  val_acc=0.5250
GRU  [Epoch 4] train_loss=0.6931  val_acc=0.5250
GRU  [Epoch 5] train_loss=0.6936  val_acc=0.4750
GRU  [Epoch 6] train_loss=0.6935  val_acc=0.5250

Ans)
T가 증가할 수록, 장기 기억에 유리한 LSTM이 GRU에 비해 성능을 잘 유지할 것 이라고 예상했지만, T를 80, 150, 300으로 증가시켜도 정확도는 거의 변화가 없이 0.5 내외로 형성이 되었다.
loss 또한 0.69정도에서 머물렀다.
두 모델 모두 장기 의존성 문제를 해결하지 못하고, 학습이 정체되어있는 것처럼 보인다.
